# Bronze local - leitura de CSVs e escrita em Delta

In [ ]:
import os
from pathlib import Path
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

java_home = Path("/usr/lib/jvm/java-17-openjdk-amd64")
if java_home.exists():
    os.environ["JAVA_HOME"] = str(java_home)
    os.environ["PATH"] = str(java_home / "bin") + os.pathsep + os.environ.get("PATH", "")
else:
    raise RuntimeError("JDK 17 não encontrado em /usr/lib/jvm/java-17-openjdk-amd64. Instale o OpenJDK 17 para rodar o Spark localmente.")

os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

project_root = Path.cwd().resolve()
for candidate in [project_root, *project_root.parents]:
    if (candidate / "landing" / "dados").exists():
        project_root = candidate
        break

caminho_landing = (project_root / "landing" / "dados").resolve()
caminho_bronze = (project_root / "bronze").resolve()
workspace_dir = (project_root / "spark-warehouse").resolve()

os.makedirs(caminho_bronze, exist_ok=True)
os.makedirs(workspace_dir, exist_ok=True)

try:
    current = SparkSession.getActiveSession()
    if current is not None:
        current.stop()
except Exception:
    pass

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("Projeto_Medalhao_Local")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.warehouse.dir", str(workspace_dir))
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark com Delta Lake iniciado com sucesso!")
print(f"Project root: {project_root}")
print(f"Landing: {caminho_landing}")
print(f"Bronze: {caminho_bronze}")

IndentationError: unexpected indent (1061078267.py, line 35)

In [ ]:
def read_csv_local(nome_arquivo: str):
    nome_base = nome_arquivo.strip()
    dir_dados = Path(caminho_landing)

    for item in sorted(dir_dados.iterdir()):
        nome_item = item.name.lower().replace(":zone.identifier", "")
        nome_sem_ext = nome_item.rsplit(".csv", 1)[0] if nome_item.endswith(".csv") else nome_item
        nome_sem_ext = nome_sem_ext.rsplit(".CSV", 1)[0] if nome_sem_ext.endswith(".CSV") else nome_sem_ext

        if nome_sem_ext == nome_base.lower():
            df = (
                spark.read
                .option("inferSchema", "true")
                .option("header", "true")
                .csv(str(item))
            )
            colunas_corrigidas = [c.replace("\ufeff", "") for c in df.columns]
            return df.toDF(*colunas_corrigidas)

    raise FileNotFoundError(f"Arquivo '{nome_arquivo}' não encontrado em {caminho_landing}")


# Leitura dos arquivos CSVs
df_teacher = read_csv_local("teacher")
df_classes = read_csv_local("classes")
df_students = read_csv_local("students")

df_subjects = read_csv_local("subjects")print("CSVs lidos com sucesso!")

df_grades = read_csv_local("grades")


## Adicionar dados de auditoria e persistir no formato Delta local

In [ ]:
from pyspark.sql.functions import current_timestamp, lit

# Adicionar metadados de processamento
df_teacher = df_teacher.withColumn("data_hora_bronze", current_timestamp()).withColumn("nome_arquivo", lit("teacher"))
df_classes = df_classes.withColumn("data_hora_bronze", current_timestamp()).withColumn("nome_arquivo", lit("classes"))
df_students = df_students.withColumn("data_hora_bronze", current_timestamp()).withColumn("nome_arquivo", lit("students"))
df_subjects = df_subjects.withColumn("data_hora_bronze", current_timestamp()).withColumn("nome_arquivo", lit("subjects"))
df_grades = df_grades.withColumn("data_hora_bronze", current_timestamp()).withColumn("nome_arquivo", lit("grades"))

# Persistência local em Delta
for nome_tabela, df in {
    "teacher": df_teacher,
    "classes": df_classes,
    "students": df_students,
    "subjects": df_subjects,
    "grades": df_grades,
}.items():
    df.write.format("delta").mode("overwrite").save(str(Path(caminho_bronze) / nome_tabela))
    print(f"Tabela salva em Delta: {nome_tabela}")
